In [ ]:
# Install required packages.
!pip install mlflow boto3 awscli optuna imbalanced-learn lightgbm


In [ ]:
# Set up AWS credentials.
!aws configure


In [ ]:
import mlflow

# Set the MLflow tracking server.
mlflow.set_tracking_uri("http://ec2-23-20-151-48.compute-1.amazonaws.com:5000/")


In [ ]:
# Set the MLflow experiment name.
mlflow.set_experiment("LightGBM HP Tuning")


In [ ]:
import pandas as pd

# Load the cleaned dataset.
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna()

# Check dataset shape.
df.shape


In [ ]:
# Import the main libraries.
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt


In [ ]:
# Remap the target labels.
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Drop rows with missing labels.
df = df.dropna(subset=['category'])


In [ ]:
# Set TF-IDF options.
ngram_range = (1, 3)
max_features = 1000

# Convert text to TF-IDF features.
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])

# Set the target column.
y = df['category']

# Balance the classes with SMOTE.
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)


In [ ]:
# Split the data into train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)


In [ ]:
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    """Train the model and log the results."""

    # Start an MLflow run.
    with mlflow.start_run():
        # Add run tags.
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log the model name.
        mlflow.log_param("algo_name", model_name)

        # Log the hyperparameters.
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train the model.
        model.fit(X_train, y_train)

        # Make predictions.
        y_pred = model.predict(X_test)

        # Log accuracy.
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Build the classification report.
        classification_rep = classification_report(y_test, y_pred, output_dict=True)

        # Log report metrics.
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Save the trained model.
        mlflow.sklearn.log_model(model, f"{model_name}_model")

        # Return the score for Optuna.
        return accuracy


In [ ]:
def objective_lightgbm(trial):
    """Optuna objective for LightGBM."""

    # Sample hyperparameters.
    n_estimators = trial.suggest_int('n_estimators', 100, 1000)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 15)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True)

    # Store the sampled values.
    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth,
        'num_leaves': num_leaves,
        'min_child_samples': min_child_samples,
        'colsample_bytree': colsample_bytree,
        'subsample': subsample,
        'reg_alpha': reg_alpha,
        'reg_lambda': reg_lambda
    }

    # Build the model.
    model = LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        colsample_bytree=colsample_bytree,
        subsample=subsample,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42
    )

    # Train and score this trial.
    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)

    return accuracy


In [ ]:
def run_optuna_experiment():
    """Run Optuna and rebuild the best model."""

    # Create the Optuna study.
    study = optuna.create_study(direction="maximize")

    # Run the tuning process.
    study.optimize(objective_lightgbm, n_trials=100)

    # Get the best parameters.
    best_params = study.best_params

    # Build the best model.
    best_model = LGBMClassifier(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        num_leaves=best_params['num_leaves'],
        min_child_samples=best_params['min_child_samples'],
        colsample_bytree=best_params['colsample_bytree'],
        subsample=best_params['subsample'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        random_state=42
    )

    # Log the best model.
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best")

    # Show Optuna plots.
    optuna.visualization.plot_param_importances(study).show()
    optuna.visualization.plot_optimization_history(study).show()

    # Return the best trained model.
    return best_model


In [ ]:
# Run the experiment and save the best model.
best_model = run_optuna_experiment()


In [ ]:
# Show the best model.
best_model
